# Fire & Recovery — Iberian Peninsula

Overlay of CLMS Burnt Area (monthly) on NDVI (10-daily / dekadal)
to visualise fire scars and vegetation recovery over the Iberian Peninsula.


In [ ]:
from functools import partial
from pathlib import Path

from rs_tools.config import BoundingBox
from rs_tools.datasets.loader import load_dataset, load_passes_from_disk
from rs_tools.visualization.animation import save_timeseries_gif_lazy
from rs_tools.visualization.frames import make_overlay_composite
from rs_tools.visualization.clms_colormaps import CLMS_NDVI, NDVI_VMIN, NDVI_VMAX, CLMS_BA

In [ ]:
bbox = BoundingBox(west=-10, south=36, east=4, north=44)
DATA_DIR = "/home/bekaertd/RS_applications/Applications/CGOPS/fire_recovery"
gif_dir = Path('output/gifs')
gif_dir.mkdir(parents=True, exist_ok=True)

# Which dekads to include: [1], [2], [3], [1,2], etc.  None = all dekads.
DEKADS = None

## Load NDVI (dekadal) and Burnt Area (monthly)

In [ ]:
ndvi_items = load_dataset(
    "CLMS_NDVI_V3", bbox=bbox,
    start_date="2020-01-01", end_date="2026-03-01",
    limit=500, output_dir=f"{DATA_DIR}/ndvi", dekads=DEKADS,
)
print(f"NDVI: {len(ndvi_items)} dekads")

In [ ]:
ba_items = load_dataset(
    "CLMS_BA_V4_MONTHLY", bbox=bbox,
    start_date="2020-01-01", end_date="2026-03-01",
    limit=500, output_dir=f"{DATA_DIR}/ba", dekads=DEKADS,
)
print(f"BA: {len(ba_items)} months")

In [ ]:
# Reload as lightweight metadata references (no pixel data in RAM)
ndvi_items = load_passes_from_disk(f"{DATA_DIR}/ndvi", dekads=DEKADS)
ba_items = load_passes_from_disk(f"{DATA_DIR}/ba", dekads=DEKADS)
n = min(len(ndvi_items), len(ba_items))
ndvi_items, ba_items = ndvi_items[:n], ba_items[:n]
print(f"NDVI: {n} dekads  BA: {n} items")

## Overlay animation (lazy — one frame at a time)

In [ ]:
overlay_composite = partial(
    make_overlay_composite,
    base_cmap=CLMS_NDVI, overlay_cmap=CLMS_BA,
    base_vmin=NDVI_VMIN, base_vmax=NDVI_VMAX,
    overlay_threshold=0.1, overlay_alpha=0.7,
)

def _composite(pair):
    base, overlay = pair
    return overlay_composite(base, overlay)

gif_path = save_timeseries_gif_lazy(
    zip(ndvi_items, ba_items),
    gif_dir / "fire_recovery_iberia.gif",
    composite_fn=_composite,
    title="Fire & Recovery — Iberian Peninsula",
    fps=6,
)
print(f"Saved: {gif_path}")